# NeSLE: verify every claimed number

Re-measures every performance claim in the README from a clean clone and prints
a claimed-vs-measured PASS / FAIL table.

All logic lives in [`benchmarks/verify_claims.py`](https://github.com/hbofz/NeSLE/blob/main/benchmarks/verify_claims.py);
this notebook only supplies a ROM and runs it. That script in turn reuses the
functions in `benchmarks/gpu_vs_cpu.py`, so the protocol matches the published
tables exactly (frameskip 4, RIGHT held, 30 warmup + 200 timed steps,
`render_frame=False, copy_obs=False`).

**Runtime:** Runtime > Change runtime type > GPU. An **A100** reproduces the
large-batch claims; a T4 or L4 runs everything else and skips the A100 rows.

**You must supply a legally obtained `Super Mario Bros. (World).nes`** (iNES,
mapper 0). No ROM ships with the repository or this notebook.

> Prefer your terminal? The same run, without a browser:
> ```
> pip install google-colab-cli
> colab new -s nesle --gpu A100
> colab upload -s nesle "Super Mario Bros. (World).nes" /content/rom.nes
> colab exec   -s nesle -f benchmarks/verify_claims.py
> colab download -s nesle /content/verification.json ./docs/data/
> colab stop -s nesle
> ```

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Get the code

In [ ]:
!git clone --depth 1 https://github.com/hbofz/NeSLE.git /content/NeSLE

## 3. Supply the ROM

Upload the file, or set `DRIVE_ROM` to a path in your mounted Drive.

In [ ]:
from pathlib import Path

ROM = Path("/content/rom.nes")
DRIVE_ROM = ""  # e.g. "/content/drive/MyDrive/mario_rl/roms/Super Mario Bros. (World).nes"

if DRIVE_ROM:
    import shutil; shutil.copy(DRIVE_ROM, ROM)
elif not ROM.exists():
    from google.colab import files
    up = files.upload()
    ROM.write_bytes(up[next(iter(up))])

print(f"{ROM}: {ROM.stat().st_size:,} bytes")

## 4. Run the verification

Installs the package, builds the CUDA extension, runs the test suite and the
falsifiability checks, sweeps throughput, measures memory at the peak batch,
and benchmarks nes-py on one core and on all cores.

Expect roughly 15 to 30 minutes. Output streams live.

In [ ]:
!python /content/NeSLE/benchmarks/verify_claims.py --repo /content/NeSLE --rom /content/rom.nes --out /content/verification.json


## 5. Keep the evidence

In [ ]:
import json
report = json.load(open("/content/verification.json"))
print(json.dumps(report["environment"], indent=2))
print("\nverdicts:")
for v in report["verdicts"]:
    print(f"  {v['envs']:>9,} envs  {v['verdict']}")

try:
    from google.colab import files; files.download("/content/verification.json")
except Exception as exc:
    print("download unavailable:", exc)

Commit the downloaded JSON to `docs/data/` so the published numbers are
checkable rather than asserted.